In [2]:
from scipy import stats
from statsmodels.stats.weightstats import ztest
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import numpy as np
import statistics as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
import random

data = pd.read_csv('nba_second_round_history_augmented.csv')
data

,Season,Team,Opponent,Games played,Average Points scored,Average Points allowed,Offensive Rating,Defensive Rating,eFG%,Opp eFG%,TOV%,Opp TOV%,ORB%,FT/FGA,Opp FT/FGA,Champion,Actual Wins (%),Team TS%,Opp TS%,DRB%
0,1984,BOS,NYK,7,111.0,103.0,113.9,105.7,49.8,48.4,14.2,16.0,38.7,28.5,29.6,True,57.143,54.9350,53.9574,65.0
1,1984,MIL,NJN,6,98.2,96.3,102.8,100.9,47.1,40.5,18.3,12.2,32.6,37.3,34.0,False,66.667,54.0129,47.1452,64.5
2,1984,LAL,DAL,5,120.6,106.2,121.9,107.4,57.0,45.4,12.8,11.9,37.8,20.0,23.2,False,80.000,59.7550,50.4063,64.9
3,1984,PHO,UTA,6,103.7,101.0,107.0,104.3,46.2,46.8,12.9,15.3,33.9,23.0,30.0,False,66.667,51.2187,52.6627,67.6
4,1985,BOS,DET,6,120.5,112.7,118.1,110.4,51.9,47.8,13.6,11.4,37.1,30.2,19.6,False,66.667,57.8882,51.6914,64.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,2024,MIN,DEN,7,102.7,97.6,113.8,108.1,53.9,51.7,11.5,12.4,22.8,20.1,17.3,False,57.143,57.2051,55.1482,78.9
164,2025,IND,CLE,5,117.6,114.2,116.7,113.3,58.5,49.0,13.3,11.7,18.0,22.0,29.5,False,80.000,61.7907,54.7480,71.2
165,2025,NYK,BOS,6,105.7,105.2,112.9,112.4,50.8,51.5,11.7,12.2,29.6,21.1,17.6,False,66.667,54.4150,54.7933,72.4
166,2025,OKC,DEN,7,115.4,106.3,114.6,105.6,51.8,48.1,9.7,15.1,24.9,20.8,24.9,True,57.143,55.5708,53.2585,72.0


"Series Score" of the advancing teams is the data viewers can most easily access through medias, compared to avarage point difffence, Net Ratings, Efficient Field Goal%, etc.
We would like to test if teams advancing with different series score have different championship proportion. Data was collected from 1984 to 2025 of the NBA playoffs second round.

In [4]:
dominant = data[data['Games played'] <= 5]
non_dominant = data[data['Games played'] > 5]

In [5]:
# check if two sided proportion z-test is appropriate
n1p1 = len(dominant[dominant['Champion'] == True])
n1q1 = len(dominant) - n1p1
n2p2 = len(non_dominant[non_dominant['Champion'] == True])
n2q2 = len(non_dominant) - n2p2
if n1p1 >= 15 and n1q1 >= 15 and n2p2 >= 15 and n2q2 >= 15:
    print("Two-sided proportion z-test is appropriate.")
if n1p1 < 15:
    print(f"Two-sided proportion z-test is not appropriate. Number of dominant champions is less than 15 ({n1p1}).")
if n1q1 < 15:
    print(f"Two-sided proportion z-test is not appropriate. Number of dominant non-champions is less than 15 ({n1q1}).")
if n2p2 < 15:
    print(f"Two-sided proportion z-test is not appropriate. Number of non-dominant champions is less than 15 ({n2p2}).")
if n2q2 < 15:
    print(f"Two-sided proportion z-test is not appropriate. Number of non-dominant non-champions is less than 15 ({n2q2}).")

Two-sided proportion z-test is appropriate.


H0: p1 = p2 ; 
H1: p1 != p2

In [ ]:
test_stat, p_value = proportions_ztest([n1p1, n2p2], [len(dominant), len(non_dominant)], alternative='two-sided')
print(f"Z-test statistic: {test_stat:.4f}")
print(f"P-value: {p_value:.4f}")
if p_value < 0.05:
    print("Reject the null hypothesis: There is a significant difference in championship rates between dominant and non-dominant teams.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in championship rates between dominant and non-dominant teams.")

Z-test statistic: 1.9740
P-value: 0.0484
Reject the null hypothesis: There is a significant difference in championship rates between dominant and non-dominant teams.


We are curious about which statistic best predicts the championship, so after standardising the data (to neutralise the influence of era), we perform **Random Forest**

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, precision_score
standard = pd.read_csv('nba_second_round_history_standardised.csv')

# Chronological split to prevent time travalling
train_data = standard[standard['Season'] < 2015]
test_data = standard[standard['Season'] >= 2015]

features = standard.drop(columns=['Season', 'Champion', 'Average Points scored', 'Average Points allowed', 'eFG%', 'Opp eFG%', 'Team TS%']).head().columns.to_list()

print("Features used for training:", features)  

x_train = train_data[features]
y_train = train_data['Champion']

x_test = test_data[features]
y_test = test_data['Champion']

model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Champion', 'Champion']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Features used for training: ['Games played', 'Offensive Rating', 'Defensive Rating', 'TOV%', 'ORB%', 'FT/FGA', 'Opp TOV%', 'DRB%', 'Opp FT/FGA']
Classification Report:
              precision    recall  f1-score   support

Non-Champion       0.81      0.91      0.86        33
    Champion       0.57      0.36      0.44        11

    accuracy                           0.77        44
   macro avg       0.69      0.64      0.65        44
weighted avg       0.75      0.77      0.75        44

Confusion Matrix:
[[30  3]
 [ 7  4]]


In [5]:
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=False)
print("Feature Importances:")
print(feature_importance_df)

Feature Importances:
            Feature  Importance
1  Offensive Rating    0.152678
2  Defensive Rating    0.150563
3              TOV%    0.118783
6          Opp TOV%    0.117748
4              ORB%    0.114455
8        Opp FT/FGA    0.110606
5            FT/FGA    0.103539
7              DRB%    0.099371
0      Games played    0.032256
